In [1]:
import pandas as pd
import numpy as np

# NPR

In [ ]:
npr_count = pd.read_excel('npr_count.xlsx', index_col=0)
npr_proportion = npr_count.div(npr_count.sum(axis=1), axis=0)

# Get the last row of npr_proportion
last_row = npr_proportion.iloc[-1]

# Create new ages from 81 to 100
new_ages = range(81, 101)

# Create a new DataFrame for these ages, using the last row's values
new_rows = pd.DataFrame([last_row.values] * len(new_ages), columns=npr_proportion.columns, index=new_ages)

new_rows.index.name = 'age'

# Append to the original DataFrame
npr_proportion_extended = pd.concat([npr_proportion, new_rows])

# Optional: sort index if needed
npr_proportion_extended = npr_proportion_extended.sort_index()
with pd.ExcelWriter('../data/tx_isq.xlsx', engine='openpyxl', mode='a', if_sheet_exists="replace") as writer:
    npr_proportion_extended.to_excel(writer,sheet_name='npr_proportion')

# Immigrants

In [35]:
immigrants_2022 = pd.read_excel('immigrants.xlsx', index_col=0)
#Read mortality data and immigration data from tx_isq
data = pd.read_excel('../data/tx_isq.xlsx', index_col=0,sheet_name='tx_by_age')
immigrants_2022

,0,1,2,3,4,5,6,7,8,9,10
0,100,100,100,100,100,100,100,100,100,100,100
1,100,100,100,100,100,100,100,100,100,100,100
2,100,100,100,100,100,100,100,100,100,100,100
3,100,100,100,100,100,100,100,100,100,100,100
4,100,100,100,100,100,100,100,100,100,100,100
...,...,...,...,...,...,...,...,...,...,...,...
86,100,100,100,100,100,100,100,100,100,100,100
87,100,100,100,100,100,100,100,100,100,100,100
88,100,100,100,100,100,100,100,100,100,100,100
89,100,100,100,100,100,100,100,100,100,100,100


In [36]:
# Get the last row of immigrants_2022
last_row = immigrants_2022.iloc[-2]
total_90p = immigrants_2022.iloc[-1]
# Create new ages from 91 to 100
new_ages = range(91, 101)

new_rows = pd.DataFrame([np.zeros(len(last_row))] * len(new_ages), columns=immigrants_2022.columns, index=new_ages)

new_rows.index.name = 'age'

# Append to the original DataFrame
immigrants_2022_extended = pd.concat([immigrants_2022, new_rows])
immigrants_2022_extended.loc[90] = last_row * (1 - data.loc[(data['year']==2023) & (data['age']==89), 'mx'].values[0])

for age in range(91, 101):
    immigrants_2022_extended.loc[age] = immigrants_2022_extended.loc[age-1] * (1 - data.loc[(data['year']==2023) & (data['age']==age-1), 'mx'].values[0])
immigrants_2022_extended.loc[90:] = immigrants_2022_extended.loc[90:] * total_90p/immigrants_2022_extended.loc[90:].sum()

In [44]:
#Bring immigrants_2022_extended to 2024
immigrants_2023 = immigrants_2022_extended.copy()

for a in range(0,101):
    for d in range(10):
        if a != 0:
            immigrants_2023.loc[a, d+1] = immigrants_2022_extended.loc[a-1, d]  #Shift existing immigrants by 1 year (duration 0→1, ..., 9→10)
    if a != 0:
        immigrants_2023.loc[a, 10] += immigrants_2022_extended.loc[a-1, 10]
        immigrants_2023.loc[a] -= immigrants_2022_extended.loc[a-1] * data.loc[(data['year']==2023) & (data['age']==a-1), 'mx'].values[0]
    immigrants_2023.loc[a,0] += data.loc[(data['year']==2023) & (data['age']==a), 'immigrant'].values[0]

immigrants_2024 = immigrants_2023.copy()

for a in range(0,101):
    for d in range(10):
        if a != 0:
            immigrants_2024.loc[a, d+1] = immigrants_2023.loc[a-1, d]  #Shift existing immigrants by 1 year (duration 0→1, ..., 9→10)
    if a != 0:
        immigrants_2024.loc[a, 10] += immigrants_2023.loc[a-1, 10]
        immigrants_2024.loc[a] -= immigrants_2023.loc[a-1] * data.loc[(data['year']==2024) & (data['age']==a-1), 'mx'].values[0]

    immigrants_2024.loc[a,0] += data.loc[(data['year']==2024) & (data['age']==a), 'immigrant'].values[0]

with pd.ExcelWriter('../data/tx_isq.xlsx', engine='openpyxl', mode='a', if_sheet_exists="replace") as writer:
    immigrants_2024.to_excel(writer,sheet_name='immig_pop')

In [45]:
immigrants_2024

,0,1,2,3,4,5,6,7,8,9,10
0,1202.803981,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,1945.484209,640.650356,99.959065,99.959065,99.959065,99.959065,99.959065,99.959065,99.959065,99.959065,199.959065
2,2137.531718,1003.871445,99.932871,99.932871,99.932871,99.932871,99.932871,99.932871,99.932871,99.932871,299.865732
3,2236.572942,1097.533306,99.956764,99.956764,99.956764,99.956764,99.956764,99.956764,99.956764,99.956764,299.913523
4,2284.148120,1145.536046,99.971643,99.971643,99.971643,99.971643,99.971643,99.971643,99.971643,99.971643,299.943284
...,...,...,...,...,...,...,...,...,...,...,...
96,18.637850,24.089268,32.964813,32.964813,32.964813,32.964813,32.964813,32.964813,32.964813,32.964813,113.977780
97,13.643124,17.885056,25.908956,25.908956,25.908956,25.908956,25.908956,25.908956,25.908956,25.908956,91.012184
98,9.776011,12.983921,19.966424,19.966424,19.966424,19.966424,19.966424,19.966424,19.966424,19.966424,71.205429
99,6.731626,9.218995,15.112685,15.112685,15.112685,15.112685,15.112685,15.112685,15.112685,15.112685,54.673679
